### Load the Air Quality Dataset and Clean It

We will read the dataset `/content/AirQualityUCI.csv`. Note that this specific dataset often uses semicolons (`;`) as delimiters and commas (`,`) as decimal separators.

In [1]:
import pandas as pd
import json

# Load the CSV file. AirQualityUCI often uses ';' as delimiter and ',' as decimal separator
try:
    df = pd.read_csv('/content/AirQualityUCI.csv', sep=';', decimal=',')
except Exception as e:
    # Fallback to default if there's an issue
    df = pd.read_csv('/content/AirQualityUCI.csv')

# Drop any completely empty columns or rows that often occur at the end of this dataset
df = df.dropna(how='all')
df = df.dropna(subset=['Date', 'Time'])

print(f"Loaded dataset with shape: {df.shape}")
display(df.head())


Loaded dataset with shape: (9357, 17)


,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,NaN,NaN
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,NaN,NaN
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,NaN,NaN
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,NaN,NaN
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,NaN,NaN


### Step 2: Convert CSV to intermediate JSON Draft File and Save It
We convert the cleaned DataFrame into a standard JSON array of objects and save it as a draft file `/content/air_quality_draft.json`. We will also display a preview of this JSON draft.

In [2]:
json_draft_path = '/content/air_quality_draft.json'

# Convert DataFrame to JSON (list of records)
json_data = df.head(100).to_dict(orient='records')

# Save the JSON draft file
with open(json_draft_path, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, indent=4, ensure_ascii=False)

print(f"Intermediate JSON draft file successfully saved at: {json_draft_path}")

# Display the first object of the saved JSON file as a preview
print("\n--- Preview of the saved JSON Draft ---")
print(json.dumps(json_data[0], indent=4))


Intermediate JSON draft file successfully saved at: /content/air_quality_draft.json

--- Preview of the saved JSON Draft ---
{
    "Date": "10/03/2004",
    "Time": "18.00.00",
    "CO(GT)": 2.6,
    "PT08.S1(CO)": 1360.0,
    "NMHC(GT)": 150.0,
    "C6H6(GT)": 11.9,
    "PT08.S2(NMHC)": 1046.0,
    "NOx(GT)": 166.0,
    "PT08.S3(NOx)": 1056.0,
    "NO2(GT)": 113.0,
    "PT08.S4(NO2)": 1692.0,
    "PT08.S5(O3)": 1268.0,
    "T": 13.6,
    "RH": 48.9,
    "AH": 0.7578,
    "Unnamed: 15": NaN,
    "Unnamed: 16": NaN
}


### Step 3: Load JSON Draft File and Convert to JSONL
Now, we load the raw JSON draft from `/content/air_quality_draft.json` and structure its contents into JSONL format suitable for training small language models.

In [3]:
jsonl_file_path = '/content/air_quality_training_data.jsonl'

# Load the JSON draft file
with open(json_draft_path, 'r', encoding='utf-8') as f:
    loaded_json_data = json.load(f)

# Generate JSONL file
with open(jsonl_file_path, 'w', encoding='utf-8') as f:
    for row in loaded_json_data:
        prompt = f"Predict the absolute humidity given the following parameters: Date: {row['Date']}, Time: {row['Time']}, CO(GT): {row['CO(GT)']}, NOx(GT): {row['NOx(GT)']}, NO2(GT): {row['NO2(GT)']}."
        completion = f" The absolute humidity is {row['AH']}."

        training_example = {
            "prompt": prompt,
            "completion": completion
        }

        f.write(json.dumps(training_example) + '\n')

print(f"JSONL training file generated successfully from the JSON draft at: {jsonl_file_path}")


JSONL training file generated successfully from the JSON draft at: /content/air_quality_training_data.jsonl


### Convert to JSON and Structure for SLM Training (JSONL)

We will format each row of our dataset into a text generation prompt-completion pair, which is standard for training small language models. Then, we will write these to a `.jsonl` file.

In [4]:
jsonl_file_path = '/content/air_quality_training_data.jsonl'

with open(jsonl_file_path, 'w', encoding='utf-8') as f:
    for index, row in df.head(100).iterrows():  # Processing a sample of 100 rows for demonstration
        # Create a prompt describing the features, and a completion representing the target (e.g., Temperature)
        prompt = f"Predict the absolute humidity given the following parameters: Date: {row['Date']}, Time: {row['Time']}, CO(GT): {row['CO(GT)']}, NOx(GT): {row['NOx(GT)']}, NO2(GT): {row['NO2(GT)']}."
        completion = f" The absolute humidity is {row['AH']}."

        # Structure according to common LLM/SLM fine-tuning schemas (prompt/completion)
        training_example = {
            "prompt": prompt,
            "completion": completion
        }

        f.write(json.dumps(training_example) + '\n')

print(f"JSONL training file generated successfully at: {jsonl_file_path}")


JSONL training file generated successfully at: /content/air_quality_training_data.jsonl


### Load and Display the JSONL Contents

Now, we load the newly created JSONL file back and display its records.

In [5]:
loaded_records = []
with open(jsonl_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        loaded_records.append(json.loads(line.strip()))

# Convert to a DataFrame to display cleanly inside the notebook
df_jsonl = pd.DataFrame(loaded_records)
display(df_jsonl.head(10))


,prompt,completion
0,Predict the absolute humidity given the follow...,The absolute humidity is 0.7578.
1,Predict the absolute humidity given the follow...,The absolute humidity is 0.7255.
2,Predict the absolute humidity given the follow...,The absolute humidity is 0.7502.
3,Predict the absolute humidity given the follow...,The absolute humidity is 0.7867.
4,Predict the absolute humidity given the follow...,The absolute humidity is 0.7888.
5,Predict the absolute humidity given the follow...,The absolute humidity is 0.7848.
6,Predict the absolute humidity given the follow...,The absolute humidity is 0.7603.
7,Predict the absolute humidity given the follow...,The absolute humidity is 0.7702.
8,Predict the absolute humidity given the follow...,The absolute humidity is 0.7648.
9,Predict the absolute humidity given the follow...,The absolute humidity is 0.7517.
